# 📊 Exploratory Data Analysis — Road Damage Dataset
**RoadSense AI | EDA Notebook**

This notebook covers:
- Dataset overview and class distribution
- Sample image visualization per class
- Bounding box statistics
- Pixel intensity analysis
- Image quality checks

In [ ]:
import os, sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from collections import Counter
import yaml

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

DATASET_PATH = config['data']['dataset_path']
IMAGE_DIR    = os.path.join(DATASET_PATH, 'data/images')
LABEL_DIR    = os.path.join(DATASET_PATH, 'data/labels')
CLASS_NAMES  = config['classes']   # {0:'Pothole', 1:'Crack', 2:'Manhole'}
print('Dataset path:', DATASET_PATH)

## 1. Dataset Overview

In [ ]:
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg','.png'))]
label_files = [f for f in os.listdir(LABEL_DIR) if f.endswith('.txt')]
print(f'Total images : {len(image_files)}')
print(f'Total labels : {len(label_files)}')

## 2. Class Distribution

In [ ]:
all_class_ids = []
dominant_labels = []

for fname in image_files:
    lbl_path = os.path.join(LABEL_DIR, fname.replace('.jpg','.txt').replace('.png','.txt'))
    if not os.path.exists(lbl_path): continue
    with open(lbl_path) as f:
        ids = [int(l.split()[0]) for l in f if l.strip()]
    all_class_ids.extend(ids)
    if ids:
        dominant_labels.append(Counter(ids).most_common(1)[0][0])

dom_counts = Counter(dominant_labels)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dominant class per image
labels = [CLASS_NAMES[k] for k in sorted(dom_counts)]
values = [dom_counts[k] for k in sorted(dom_counts)]
axes[0].bar(labels, values, color=['#e94560','#4a90d9','#48bb78'])
axes[0].set_title('Dominant Class Distribution (per image)')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(values, labels=labels, autopct='%1.1f%%',
            colors=['#e94560','#4a90d9','#48bb78'], startangle=140)
axes[1].set_title('Class Share')
plt.tight_layout()
plt.savefig('../reports/figures/eda_class_distribution.png', dpi=150)
plt.show()

## 3. Sample Images per Class

In [ ]:
from src.data_preparation import load_dataset
image_paths, labels = load_dataset(DATASET_PATH)

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for cls_id in range(3):
    cls_paths = [p for p, l in zip(image_paths, labels) if l == cls_id][:5]
    for j, path in enumerate(cls_paths):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        axes[cls_id][j].imshow(img)
        axes[cls_id][j].axis('off')
        if j == 0:
            axes[cls_id][j].set_ylabel(CLASS_NAMES[cls_id], fontsize=13, fontweight='bold')
plt.suptitle('Sample Images per Damage Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/eda_sample_images.png', dpi=150)
plt.show()

## 4. Pixel Intensity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'Pothole':'#e94560', 'Crack':'#4a90d9', 'Manhole':'#48bb78'}
for cls_id in range(3):
    cls_paths = [p for p, l in zip(image_paths, labels) if l == cls_id][:50]
    all_pixels = []
    for path in cls_paths:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            all_pixels.extend(img.flatten().tolist())
    ax.hist(all_pixels, bins=64, alpha=0.5, label=CLASS_NAMES[cls_id],
            color=colors[CLASS_NAMES[cls_id]], density=True)
ax.set_xlabel('Pixel Intensity')
ax.set_ylabel('Density')
ax.set_title('Pixel Intensity Distribution by Class')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/eda_pixel_intensity.png', dpi=150)
plt.show()

## 5. Bounding Box Statistics

In [ ]:
bbox_data = []
for fname in image_files[:500]:
    lbl_path = os.path.join(LABEL_DIR, fname.replace('.jpg','.txt').replace('.png','.txt'))
    if not os.path.exists(lbl_path): continue
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
                bbox_data.append({'class': CLASS_NAMES.get(cls_id,'Unknown'),
                                  'cx':cx,'cy':cy,'w':w,'h':h,'area':w*h})

df_bbox = pd.DataFrame(bbox_data)
print(df_bbox.groupby('class')[['w','h','area']].describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cls, grp in df_bbox.groupby('class'):
    axes[0].scatter(grp['w'], grp['h'], alpha=0.3, label=cls, s=10)
axes[0].set_xlabel('BBox Width (normalized)')
axes[0].set_ylabel('BBox Height (normalized)')
axes[0].set_title('Bounding Box Dimensions')
axes[0].legend()

df_bbox.boxplot(column='area', by='class', ax=axes[1])
axes[1].set_title('BBox Area Distribution by Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Area (normalized)')
plt.tight_layout()
plt.savefig('../reports/figures/eda_bbox_stats.png', dpi=150)
plt.show()